[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/01_sample_spaces_and_probability_axioms/exercises.ipynb)

# Exercises — Module 01: Sample Spaces and Probability Axioms

20 fully solved problems in four tiers: L0 Concept Checks (3), L1 Foundations (6), L2 Applications in AI/ML and Physics (6), L3 Challenge Proofs (5).

Every numeric answer below is recomputed in the code cell that follows it, so no boxed number rests on memory. Theorem and proof numbers refer to [`first_principles.ipynb`](first_principles.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

import math
from fractions import Fraction
from itertools import combinations

## L0 — Concept Checks

### Problem L0.1 — Probability Zero Versus Impossible

**Statement.** Let $X$ be a uniform random point on $[0,1]$. Is the event $A = \{X = 0.5\}$ impossible? What is $\mathbb{P}(A)$?

**Intuition.** A single point has no length, and probability on $[0,1]$ *is* length; but "no length" is not the same as "no member".

**Solution.**

*Step 1.* $A \ne \emptyset$, because $0.5 \in [0,1]$ is a legitimate outcome of the experiment. So $A$ is not the impossible event.

*Step 2.* For every $\epsilon \gt 0$ we have $A \subseteq [0.5-\epsilon,\, 0.5+\epsilon]$, so monotonicity (Theorem 4.2) gives $\mathbb{P}(A) \le 2\epsilon$.

*Step 3.* A non-negative number below $2\epsilon$ for every $\epsilon \gt 0$ is $0$. Hence $\mathbb{P}(A) = 0$.

*Step 4.* There is no contradiction with "some outcome always occurs": Axiom 3 licenses summing over *countably* many disjoint events, and $[0,1]$ is uncountable, so the point probabilities may not be summed over all of $\Omega$.

$$
\boxed{\mathbb{P}(X = 0.5) = 0 \quad\text{yet}\quad \{X = 0.5\} \ne \emptyset}
$$

**Key takeaway.** "Probability zero" and "impossible" coincide only on countable sample spaces; on continuous spaces they come apart.

In [2]:
# The squeeze of Step 2, made numeric: P(|X - 0.5| <= eps) = 2*eps shrinks to 0.
eps_grid = np.array([1e-1, 1e-2, 1e-3, 1e-4, 1e-5])
n = 2_000_000
X = rng.random(n)
for eps in eps_grid:
    emp = np.mean(np.abs(X - 0.5) <= eps)
    print(f"eps={eps:.0e}   exact 2*eps={2*eps:.0e}   empirical={emp:.6f}")
print(f"exact hits of X == 0.5 in {n:,} draws: {int(np.sum(X == 0.5))}")
assert np.sum(X == 0.5) == 0

eps=1e-01   exact 2*eps=2e-01   empirical=0.200308
eps=1e-02   exact 2*eps=2e-02   empirical=0.019908
eps=1e-03   exact 2*eps=2e-03   empirical=0.001977
eps=1e-04   exact 2*eps=2e-04   empirical=0.000206
eps=1e-05   exact 2*eps=2e-05   empirical=0.000022
exact hits of X == 0.5 in 2,000,000 draws: 0


### Problem L0.2 — The Complement Rule Is Forced

**Statement.** Using only the Kolmogorov axioms, show that no probability measure can have $\mathbb{P}(A) = 0.7$ and $\mathbb{P}(A^c) = 0.4$ simultaneously.

**Intuition.** $A$ and $A^c$ tile $\Omega$ exactly once, so their probabilities must spend the whole budget of 1 and no more.

**Solution.**

*Step 1.* $A$ and $A^c$ are disjoint and $A \cup A^c = \Omega$.

*Step 2.* Lemma 4.1 (finite additivity) and Axiom 2 give $\mathbb{P}(A) + \mathbb{P}(A^c) = \mathbb{P}(\Omega) = 1$.

*Step 3.* The proposed pair sums to $0.7 + 0.4 = 1.1 \ne 1$, so it violates the axioms. In betting terms it admits a Dutch book: a portfolio of bets that loses for the quoter with certainty.

$$
\boxed{\mathbb{P}(A) + \mathbb{P}(A^c) = 1 \ \text{always; the pair } (0.7,\,0.4) \text{ is inconsistent}}
$$

**Key takeaway.** Complementary probabilities are a hard constraint derived from additivity plus normalization, never an extra assumption.

In [3]:
print(f"0.7 + 0.4 = {0.7 + 0.4:.4f}   required: 1.0   excess = {0.7 + 0.4 - 1:.4f}")
assert not math.isclose(0.7 + 0.4, 1.0)

0.7 + 0.4 = 1.1000   required: 1.0   excess = 0.1000


### Problem L0.3 — Monotonicity Sanity Check

**Statement.** A colleague reports that a network packet is corrupted *and* late with probability $0.2$, while it is corrupted with probability $0.15$. Is this possible?

**Intuition.** Adding a requirement can only shrink the set of outcomes that satisfy it.

**Solution.**

*Step 1.* Let $C$ = "corrupted" and $L$ = "late". Then $C \cap L \subseteq C$.

*Step 2.* Monotonicity (Theorem 4.2) forces $\mathbb{P}(C \cap L) \le \mathbb{P}(C)$.

*Step 3.* The report asserts $0.2 \le 0.15$, which is false, so the assignment is impossible. This is the *conjunction fallacy* of Tversky and Kahneman's "Linda problem".

$$
\boxed{\text{Impossible: } \mathbb{P}(C \cap L) \le \mathbb{P}(C) \text{ would require } 0.2 \le 0.15}
$$

**Key takeaway.** Adding conditions can only shrink an event, hence can never raise its probability.

In [4]:
print(f"claimed P(C and L) = 0.20   claimed P(C) = 0.15   monotonicity holds? {0.20 <= 0.15}")
assert not (0.20 <= 0.15)

claimed P(C and L) = 0.20   claimed P(C) = 0.15   monotonicity holds? False


## L1 — Foundations

### Problem L1.1 — Inclusion–Exclusion from the Axioms

**Statement.** Prove $\mathbb{P}(A \cup B) = \mathbb{P}(A) + \mathbb{P}(B) - \mathbb{P}(A \cap B)$ from the axioms, then compute the probability that a card drawn from a standard 52-card deck is a heart or a face card (J, Q, K).

**Intuition.** Adding $\mathbb{P}(A)$ and $\mathbb{P}(B)$ counts the overlap twice; subtract it once.

**Solution.**

*Step 1.* Decompose into pairwise disjoint pieces: $A \cup B = (A\setminus B) \cup (A\cap B) \cup (B\setminus A)$. Lemma 4.1 gives $\mathbb{P}(A\cup B) = \mathbb{P}(A\setminus B) + \mathbb{P}(A\cap B) + \mathbb{P}(B\setminus A)$.

*Step 2.* The same lemma on $A = (A\setminus B)\cup(A\cap B)$ and $B = (B\setminus A)\cup(A\cap B)$ gives $\mathbb{P}(A\setminus B) = \mathbb{P}(A)-\mathbb{P}(A\cap B)$ and $\mathbb{P}(B\setminus A) = \mathbb{P}(B)-\mathbb{P}(A\cap B)$.

*Step 3.* Substituting into Step 1 yields $\mathbb{P}(A\cup B) = \mathbb{P}(A)+\mathbb{P}(B)-\mathbb{P}(A\cap B)$. $\blacksquare$

*Step 4.* Application. Under the uniform measure on 52 cards let $H$ = heart ($13$ cards), $F$ = face card ($12$ cards); $H \cap F$ = the three face hearts. Then

$$
\mathbb{P}(H\cup F) = \frac{13}{52} + \frac{12}{52} - \frac{3}{52} = \frac{22}{52} = \frac{11}{26}.
$$

$$
\boxed{\mathbb{P}(H \cup F) = \frac{11}{26} \approx 0.4231}
$$

**Key takeaway.** Overlapping events double-count their intersection; inclusion–exclusion removes it exactly once.

In [5]:
ranks = list(range(13))          # 0..12, with 10,11,12 = J,Q,K
suits = list(range(4))           # 0 = hearts
deck = [(r, s) for r in ranks for s in suits]
H = {c for c in deck if c[1] == 0}
F = {c for c in deck if c[0] >= 10}
Pd = lambda S: Fraction(len(S), len(deck))
print(f"|deck|={len(deck)}  |H|={len(H)}  |F|={len(F)}  |H&F|={len(H & F)}")
print(f"P(H u F) direct       = {Pd(H | F)} = {float(Pd(H | F)):.4f}")
print(f"P(H)+P(F)-P(H n F)    = {Pd(H) + Pd(F) - Pd(H & F)}")
assert Pd(H | F) == Pd(H) + Pd(F) - Pd(H & F) == Fraction(11, 26)

|deck|=52  |H|=13  |F|=12  |H&F|=3
P(H u F) direct       = 11/26 = 0.4231
P(H)+P(F)-P(H n F)    = 11/26


### Problem L1.2 — Die Events, De Morgan, and the Derived Rules

**Statement.** Roll a fair die. With $A = \{2,4,6\}$ (even) and $B = \{5,6\}$ (at least 5), compute $\mathbb{P}(A)$, $\mathbb{P}(B)$, $\mathbb{P}(A\cap B)$, $\mathbb{P}(A\cup B)$ and $\mathbb{P}(A^c \cap B^c)$, verifying De Morgan's law two ways.

**Intuition.** On a finite equally-likely space every probability is a count over 6, so each rule can be checked twice: symbolically and by listing outcomes.

**Solution.**

*Step 1.* Uniform measure on $\Omega = \{1,\ldots,6\}$ (Definition 3.6): $\mathbb{P}(A) = 3/6 = 1/2$ and $\mathbb{P}(B) = 2/6 = 1/3$.

*Step 2.* $A \cap B = \{6\}$, so $\mathbb{P}(A\cap B) = 1/6$.

*Step 3.* Inclusion–exclusion (Theorem 4.3): $\mathbb{P}(A\cup B) = \tfrac12 + \tfrac13 - \tfrac16 = \tfrac{3+2-1}{6} = \tfrac{2}{3}$.

*Step 4.* De Morgan gives $A^c \cap B^c = (A\cup B)^c$, so the complement rule (Theorem 4.2) gives $\mathbb{P}(A^c\cap B^c) = 1 - \tfrac23 = \tfrac13$. Direct check: $A^c \cap B^c = \{1,3\}$, of size 2, so $2/6 = 1/3$. Both routes agree.

$$
\boxed{\mathbb{P}(A\cup B) = \frac{2}{3}, \qquad \mathbb{P}(A^c \cap B^c) = \frac{1}{3}}
$$

**Key takeaway.** Complement plus De Morgan turns a union computation into an intersection computation, which is usually the easier one.

In [6]:
Om = frozenset(range(1, 7))
P6 = lambda S: Fraction(len(S), 6)
A, B = frozenset({2, 4, 6}), frozenset({5, 6})
print(f"P(A)={P6(A)}  P(B)={P6(B)}  P(A n B)={P6(A & B)}  P(A u B)={P6(A | B)}")
print(f"1 - P(A u B) = {1 - P6(A | B)}   P(A^c n B^c) direct = {P6((Om - A) & (Om - B))}"
      f"   set = {sorted((Om - A) & (Om - B))}")
assert P6(A | B) == P6(A) + P6(B) - P6(A & B) == Fraction(2, 3)
assert P6((Om - A) & (Om - B)) == 1 - P6(A | B) == Fraction(1, 3)

P(A)=1/2  P(B)=1/3  P(A n B)=1/6  P(A u B)=2/3
1 - P(A u B) = 1/3   P(A^c n B^c) direct = 1/3   set = [1, 3]


### Problem L1.3 — The Birthday Problem

**Statement.** In a room of $n = 23$ people with birthdays uniform over 365 days and independent, what is the probability that at least two share a birthday?

**Intuition.** Count the complement: 23 people generate $\binom{23}{2} = 253$ pairs, and each pair collides with probability $1/365$, so the expected number of collisions is already near 0.7.

**Solution.**

*Step 1.* Work with $A^c$ = "all $n$ birthdays distinct". The sample space of ordered birthday assignments has $\lvert\Omega\rvert = 365^n$ elements; the injective assignments number $365\cdot364\cdots(365-n+1)$.

*Step 2.* By Definition 3.6,

$$
\mathbb{P}(A^c) = \prod_{k=0}^{n-1}\frac{365-k}{365} = \prod_{k=0}^{n-1}\left(1 - \frac{k}{365}\right).
$$

*Step 3.* A quick estimate with $\ln(1-x)\approx -x$: $\ln \mathbb{P}(A^c)\approx -\tfrac{1}{365}\sum_{k=0}^{22}k = -\tfrac{253}{365}\approx-0.693$, so $\mathbb{P}(A^c)\approx e^{-0.693}\approx 0.500$.

*Step 4.* The exact product gives $\mathbb{P}(A^c) = 0.49270$, so by the complement rule $\mathbb{P}(A) = 1 - 0.49270 = 0.50730$.

$$
\boxed{\mathbb{P}(\text{shared birthday among }23) = 0.5073}
$$

**Key takeaway.** It is the number of *pairs* $\binom{n}{2}$, not $n$, that drives collisions — the same quadratic effect that governs hash-table collisions.

In [7]:
def p_shared(n, days=365):
    distinct = 1.0
    for k in range(n):
        distinct *= (days - k) / days
    return 1 - distinct

exact23 = p_shared(23)
approx23 = 1 - math.exp(-math.comb(23, 2) / 365)
trials = 200_000
sims = rng.integers(0, 365, size=(trials, 23))
mc23 = np.mean([len(np.unique(row)) < 23 for row in sims])
print(f"exact              P(shared, n=23) = {exact23:.6f}")
print(f"pair approximation 1-exp(-C(23,2)/365) = {approx23:.6f}")
print(f"Monte Carlo ({trials:,} rooms)          = {mc23:.6f}")
print(f"smallest n with P >= 1/2: {min(n for n in range(1, 80) if p_shared(n) >= 0.5)}")
assert abs(exact23 - 0.5073) < 5e-5
assert abs(mc23 - exact23) < 0.01

exact              P(shared, n=23) = 0.507297
pair approximation 1-exp(-C(23,2)/365) = 0.500002
Monte Carlo (200,000 rooms)          = 0.505830
smallest n with P >= 1/2: 23


### Problem L1.4 — Boole's Inequality and a Reliability Budget

**Statement.** Prove the union bound for finitely many events, then apply it: a pipeline runs 40 validation checks, each failing with probability at most $0.001$. Bound the probability that at least one check fails.

**Intuition.** Risk can be budgeted additively across components; the bound never undershoots, and it costs nothing in assumptions.

**Solution.**

*Step 1.* Induction on $n$. The case $n = 1$ is an equality.

*Step 2.* Assume the bound for $n-1$ events. Theorem 4.3 with two events (the union of the first $n-1$, and $A_n$) plus Axiom 1 applied to the intersection term gives

$$
\mathbb{P}\!\left(\bigcup_{i=1}^{n}A_i\right) = \mathbb{P}\!\left(\bigcup_{i=1}^{n-1}A_i\right) + \mathbb{P}(A_n) - \mathbb{P}\!\left(A_n \cap \bigcup_{i \lt n}A_i\right) \le \sum_{i=1}^{n-1}\mathbb{P}(A_i) + \mathbb{P}(A_n). \qquad\blacksquare
$$

*Step 3.* Application. With $F_i$ = "check $i$ fails" and $\mathbb{P}(F_i)\le0.001$, Theorem 4.4 gives $\mathbb{P}(\bigcup_{i=1}^{40}F_i)\le 40\times0.001 = 0.04$, with **no** independence assumption.

*Step 4.* For calibration only: if the checks *were* independent with $\mathbb{P}(F_i) = 0.001$ exactly, the true value is $1-0.999^{40} = 0.039230$, so the bound is 1.96% high.

$$
\boxed{\mathbb{P}(\text{any check fails}) \le 0.04}
$$

**Key takeaway.** The union bound trades tightness for universality — it survives arbitrary dependence and is nearly exact when the events are rare.

In [8]:
m_checks, p_fail = 40, 0.001
bound = m_checks * p_fail
exact_indep = 1 - (1 - p_fail) ** m_checks
print(f"union bound              = {bound:.6f}")
print(f"exact under independence = {exact_indep:.6f}")
print(f"bound / exact            = {bound / exact_indep:.4f}   ({bound / exact_indep - 1:.2%} high)")
assert exact_indep <= bound
assert abs(exact_indep - 0.039230) < 1e-6

union bound              = 0.040000
exact under independence = 0.039230
bound / exact            = 1.0196   (1.96% high)


### Problem L1.5 — Counting: Committees and Full Houses

**Statement.** (a) How many committees of 4 can be formed from 10 people, and what is the probability that a uniformly random committee contains a designated person? (b) What is the probability that a 5-card poker hand is a full house (three of one rank, two of another)?

**Intuition.** Both are the classical rule of Definition 3.6: build the favourable outcomes stage by stage and divide by the total count.

**Solution.**

*Step 1 (a).* Total committees: $\binom{10}{4} = 210$. Committees containing the designated person $X$: choose the other three from the remaining nine, $\binom{9}{3} = 84$.

*Step 2 (a).* $\mathbb{P}(X \in \text{committee}) = 84/210 = 2/5 = 0.4$, matching the symmetry argument $4/10$ (every seat is equally likely to hold any person).

*Step 3 (b).* Total hands: $\binom{52}{5} = 2{,}598{,}960$.

*Step 4 (b).* Build a full house in four independent stages: triple rank ($13$), its suits ($\binom43 = 4$), pair rank ($12$ remaining), its suits ($\binom42 = 6$). So $N = 13\cdot4\cdot12\cdot6 = 3744$ and $\mathbb{P} = 3744/2598960 = 0.00144058$.

$$
\boxed{\mathbb{P}(\text{designated member}) = 0.4, \qquad \mathbb{P}(\text{full house}) = 1.4406\times10^{-3}}
$$

**Key takeaway.** Structured counting implements the multiplication principle; always confirm with an independent symmetry argument when one exists.

In [9]:
p_member = Fraction(math.comb(9, 3), math.comb(10, 4))
n_fh = 13 * math.comb(4, 3) * 12 * math.comb(4, 2)
p_fh = Fraction(n_fh, math.comb(52, 5))
print(f"C(10,4)={math.comb(10,4)}  C(9,3)={math.comb(9,3)}  P(member)={p_member} = {float(p_member)}")
print(f"C(52,5)={math.comb(52,5):,}  full houses={n_fh}  P={float(p_fh):.8f} = {float(p_fh):.4e}")
assert p_member == Fraction(2, 5) == Fraction(4, 10)
assert n_fh == 3744 and abs(float(p_fh) - 1.4406e-3) < 1e-7

C(10,4)=210  C(9,3)=84  P(member)=2/5 = 0.4
C(52,5)=2,598,960  full houses=3744  P=0.00144058 = 1.4406e-03


### Problem L1.6 — Continuity of Measure in Action

**Statement.** Let $X$ be a real-valued random variable. Using continuity of probability measures, prove $\lim_{n\to\infty}\mathbb{P}(X \gt n) = 0$.

**Intuition.** The tails $\{X \gt n\}$ shrink to the empty event, and Theorem 4.5 lets probability follow them down.

**Solution.**

*Step 1.* Put $B_n = \{X \gt n\}$. Since $X \gt n+1$ implies $X \gt n$, the sequence decreases: $B_1 \supseteq B_2 \supseteq \cdots$.

*Step 2.* $\bigcap_{n\ge1}B_n = \{X \gt n \text{ for every }n\} = \{X = +\infty\} = \emptyset$, because $X$ is real-valued.

*Step 3.* Continuity from above (Theorem 4.5, decreasing case) and Lemma 4.1 give

$$
\lim_{n\to\infty}\mathbb{P}(B_n) = \mathbb{P}\!\left(\bigcap_{n=1}^{\infty}B_n\right) = \mathbb{P}(\emptyset) = 0. \qquad\blacksquare
$$

*Step 4.* The finiteness of $\mathbb{P}$ is doing real work in Step 3: the decreasing case of Theorem 4.5 needs $\mathbb{P}(B_1) \lt \infty$, automatic here and false for, say, Lebesgue measure on $\mathbb{R}$.

$$
\boxed{\lim_{n\to\infty}\mathbb{P}(X \gt n) = 0}
$$

**Key takeaway.** Continuity of measure — the payoff of *countable* additivity — is what makes tails vanish and CDFs approach 1; finite additivity alone cannot deliver it.

In [10]:
# Two laws with very different tails; both must give P(X > n) -> 0.
ns = np.arange(1, 13)
tail_normal = [float(np.mean(rng.standard_normal(400_000) > n)) for n in ns[:6]]
tail_cauchy = 0.5 - np.arctan(ns) / np.pi          # exact standard-Cauchy tail
print("standard normal, empirical P(X > n):")
print("   " + "  ".join(f"n={n}:{t:.5f}" for n, t in zip(ns[:6], tail_normal)))
print("standard Cauchy, exact P(X > n) = 1/2 - arctan(n)/pi:")
print("   " + "  ".join(f"n={n}:{t:.5f}" for n, t in zip(ns, tail_cauchy)))
assert tail_normal[-1] <= tail_normal[0] and tail_normal[-1] < 1e-4
assert np.all(np.diff(tail_cauchy) < 0) and tail_cauchy[-1] < 0.03

standard normal, empirical P(X > n):
   n=1:0.15866  n=2:0.02289  n=3:0.00135  n=4:0.00003  n=5:0.00000  n=6:0.00000
standard Cauchy, exact P(X > n) = 1/2 - arctan(n)/pi:
   n=1:0.25000  n=2:0.14758  n=3:0.10242  n=4:0.07798  n=5:0.06283  n=6:0.05257  n=7:0.04517  n=8:0.03958  n=9:0.03522  n=10:0.03173  n=11:0.02886  n=12:0.02646


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — The Union Bound in PAC Learning

**Statement.** A finite hypothesis class $\mathcal{H}$ with $\lvert\mathcal{H}\rvert = 10^6$ is evaluated on $n$ i.i.d. samples. For a single hypothesis Hoeffding's inequality gives $\mathbb{P}(\lvert\hat R(h) - R(h)\rvert \gt \epsilon) \le 2e^{-2n\epsilon^2}$. Bound the probability that *any* hypothesis deviates by more than $\epsilon = 0.05$ when $n = 5000$, and interpret.

**Intuition.** One bad hypothesis is unlikely; a million chances to be unlucky multiply the risk by a million, which an exponential absorbs cheaply.

**Solution.**

*Step 1.* Let $A_h$ be the bad event for hypothesis $h$. Theorem 4.4 over the $\lvert\mathcal{H}\rvert$ events gives

$$
\mathbb{P}\!\left(\bigcup_{h\in\mathcal{H}}A_h\right) \le \sum_{h\in\mathcal{H}}\mathbb{P}(A_h) \le \lvert\mathcal{H}\rvert\cdot 2e^{-2n\epsilon^2}.
$$

*Step 2.* With $n = 5000$ and $\epsilon = 0.05$ the exponent is $2\cdot5000\cdot0.0025 = 25$, and $e^{-25} = 1.3888\times10^{-11}$.

*Step 3.* Hence the bound is $10^{6}\cdot 2\cdot1.3888\times10^{-11} = 2.7776\times10^{-5}$.

*Step 4.* Reading it back: with probability at least $1 - 2.78\times10^{-5}$, *every* hypothesis has empirical risk within $0.05$ of its true risk simultaneously, so picking the empirical minimizer is safe. Solving $\lvert\mathcal{H}\rvert 2e^{-2n\epsilon^2}\le\delta$ for $n$ shows the sample cost grows only as $n = O\!\left(\epsilon^{-2}\ln(\lvert\mathcal{H}\rvert/\delta)\right)$.

$$
\boxed{\mathbb{P}(\text{some hypothesis deviates by} \gt 0.05) \le 2.7776\times10^{-5}}
$$

**Key takeaway.** Boole's inequality upgrades a per-hypothesis guarantee to a uniform one at only logarithmic sample cost — the axiom-level engine of generalization theory.

In [11]:
H_size, n_samp, eps = 10 ** 6, 5000, 0.05
single = 2 * math.exp(-2 * n_samp * eps ** 2)
uniform_bound = H_size * single
n_needed = math.ceil(math.log(2 * H_size / 0.05) / (2 * eps ** 2))
print(f"exponent 2*n*eps^2   = {2 * n_samp * eps ** 2:.4f}")
print(f"single-hypothesis    = {single:.6e}")
print(f"union bound over |H| = {uniform_bound:.6e}")
print(f"n needed for delta=0.05: {n_needed}")
assert abs(uniform_bound - 2.7776e-5) < 1e-9

exponent 2*n*eps^2   = 25.0000
single-hypothesis    = 2.777589e-11
union bound over |H| = 2.777589e-05
n needed for delta=0.05: 3501


### Problem L2.2 — Bonferroni Correction for Model Comparisons

**Statement.** You A/B test $m = 20$ model variants against a baseline, each with a significance test at level $\alpha_0$. (a) Bound the family-wise error rate (FWER) when all null hypotheses are true. (b) Which per-test level $\alpha_0$ guarantees FWER at most $0.05$?

**Intuition.** Every extra comparison is another chance to be fooled; the union bound says those chances add, so the per-test budget must be divided.

**Solution.**

*Step 1 (a).* Let $E_i$ be "test $i$ falsely rejects"; under the null $\mathbb{P}(E_i)\le\alpha_0$. Theorem 4.4 gives $\text{FWER} = \mathbb{P}(\bigcup_{i=1}^{20}E_i)\le 20\alpha_0$.

*Step 2 (a).* The tests share one evaluation set, so they are dependent — and that is exactly why the union bound is the right tool: it assumes nothing about dependence.

*Step 3 (b).* Requiring $20\alpha_0\le0.05$ gives $\alpha_0 = 0.05/20 = 0.0025$.

*Step 4.* For scale: with an uncorrected $\alpha_0 = 0.05$ and (hypothetically) independent tests, the FWER would be $1-0.95^{20} = 0.641514$ — a 64% chance of at least one spurious "winner".

$$
\boxed{\alpha_0 = \frac{0.05}{20} = 0.0025, \qquad \text{uncorrected FWER} = 0.6415}
$$

**Key takeaway.** Every extra comparison spends probability budget; Bonferroni divides $\alpha$ by the number of looks, using the axioms alone and no distributional assumption.

In [12]:
m_tests, fwer_target = 20, 0.05
alpha0 = fwer_target / m_tests
uncorrected = 1 - (1 - 0.05) ** m_tests
print(f"corrected per-test level alpha0 = {alpha0}")
print(f"union bound on FWER at alpha0   = {m_tests * alpha0:.4f}  (target {fwer_target})")
print(f"uncorrected FWER, independent   = {uncorrected:.6f}")
assert math.isclose(m_tests * alpha0, fwer_target)
assert abs(uncorrected - 0.641514) < 1e-6

corrected per-test level alpha0 = 0.0025
union bound on FWER at alpha0   = 0.0500  (target 0.05)
uncorrected FWER, independent   = 0.641514


### Problem L2.3 — Microstates and Entropy of a Spin System (Physics)

**Statement.** A system of $N = 100$ non-interacting spins has sample space $\Omega = \{\uparrow,\downarrow\}^{100}$ with all microstates equally likely. (a) Compute $\lvert\Omega\rvert$ and the Boltzmann entropy $S/k_B = \ln\lvert\Omega\rvert$. (b) What is the probability that exactly half the spins are up, and how does it compare with the Stirling estimate?

**Intuition.** Every microstate is equally likely, but macrostates are not: the half-and-half macrostate contains overwhelmingly the most microstates.

**Solution.**

*Step 1 (a).* Each spin has 2 states, so $\lvert\Omega\rvert = 2^{100} = 1.2677\times10^{30}$.

*Step 2 (a).* $S/k_B = \ln 2^{100} = 100\ln 2 = 69.3147$.

*Step 3 (b).* The macrostate "exactly 50 up" contains $\binom{100}{50} = 1.00891\times10^{29}$ microstates, so by Definition 3.6

$$
\mathbb{P}(50\text{ up}) = \frac{\binom{100}{50}}{2^{100}} = \frac{1.00891\times10^{29}}{1.26765\times10^{30}} = 0.079589.
$$

*Step 4.* Stirling's approximation $\binom{2n}{n}\approx 4^n/\sqrt{\pi n}$ with $n = 50$ gives $\mathbb{P}\approx 1/\sqrt{50\pi} = 0.079788$, agreeing to $0.25\%$. Although each microstate carries the same weight $2^{-100}$, the mode of the macrostate distribution sits at 50 up, which is why bulk magnetization concentrates near zero.

$$
\boxed{S/k_B = 100\ln2 = 69.3147, \qquad \mathbb{P}(50\text{ up}) = 0.079589}
$$

**Key takeaway.** Statistical mechanics is the classical uniform measure on an exponentially large sample space; macroscopic regularity is a counting fact about macrostates.

In [13]:
N = 100
omega = 2 ** N
entropy = N * math.log(2)
p_half = math.comb(N, N // 2) / omega
stirling = 1 / math.sqrt((N // 2) * math.pi)
print(f"|Omega| = 2^100      = {float(omega):.4e}")
print(f"S/k_B   = 100 ln 2   = {entropy:.4f}")
print(f"C(100,50)            = {float(math.comb(N, N//2)):.5e}")
print(f"P(50 up) exact       = {p_half:.6f}")
print(f"Stirling 1/sqrt(50 pi) = {stirling:.6f}   relative gap = {abs(stirling/p_half - 1):.4%}")
assert abs(entropy - 69.3147) < 1e-4
assert abs(p_half - 0.079589) < 1e-6
assert abs(stirling / p_half - 1) < 0.005

|Omega| = 2^100      = 1.2677e+30
S/k_B   = 100 ln 2   = 69.3147
C(100,50)            = 1.00891e+29
P(50 up) exact       = 0.079589
Stirling 1/sqrt(50 pi) = 0.079788   relative gap = 0.2503%


### Problem L2.4 — Reliability: Exact Inclusion–Exclusion Versus the Union Bound

**Statement.** A service depends on three independent subsystems that fail per day with probabilities $p_1 = 0.01$, $p_2 = 0.02$, $p_3 = 0.03$. Compute the exact probability of at least one failure by inclusion–exclusion, and compare with the union bound.

**Intuition.** With rare, near-independent failures the second-order corrections are two orders of magnitude smaller than the first-order sum, so the union bound is nearly exact.

**Solution.**

*Step 1.* Independence makes intersections factor: $\mathbb{P}(F_i\cap F_j) = p_ip_j$ and $\mathbb{P}(F_1\cap F_2\cap F_3) = p_1p_2p_3$.

*Step 2.* Theorem 4.3 with $n = 3$ gives $\mathbb{P}(F_1\cup F_2\cup F_3) = \sum_i p_i - \sum_{i\lt j}p_ip_j + p_1p_2p_3$.

*Step 3.* Numerically $\sum_i p_i = 0.06$; the pairwise sum is $0.0002+0.0003+0.0006 = 0.0011$; the triple term is $6\times10^{-6}$. So $\mathbb{P} = 0.06 - 0.0011 + 0.000006 = 0.058906$.

*Step 4.* Cross-check by the complement rule: $1 - 0.99\cdot0.98\cdot0.97 = 1 - 0.941094 = 0.058906$. The union bound gives $0.06$, high by $1.094\times10^{-3}$, a relative error of $1.86\%$.

$$
\boxed{\mathbb{P}(\text{at least one failure}) = 0.058906 \le 0.06 \text{ (union bound)}}
$$

**Key takeaway.** For small failure probabilities the union bound is essentially exact, because the corrections it discards are second order.

In [14]:
ps = np.array([0.01, 0.02, 0.03])
S1 = ps.sum()
S2 = sum(a * b for a, b in combinations(ps, 2))
S3 = float(np.prod(ps))
ie = S1 - S2 + S3
complement = 1 - float(np.prod(1 - ps))
print(f"S1={S1:.6f}  S2={S2:.6f}  S3={S3:.9f}")
print(f"inclusion-exclusion = {ie:.6f}")
print(f"1 - prod(1-p_i)     = {complement:.6f}")
print(f"union bound         = {S1:.6f}   slack = {S1 - ie:.6f}  ({S1/ie - 1:.2%} high)")
assert abs(ie - complement) < 1e-12 and abs(ie - 0.058906) < 1e-9

S1=0.060000  S2=0.001100  S3=0.000006000
inclusion-exclusion = 0.058906
1 - prod(1-p_i)     = 0.058906
union bound         = 0.060000   slack = 0.001094  (1.86% high)


### Problem L2.5 — Token Distributions and Nucleus Sampling

**Statement.** A language model outputs a probability measure $p$ on a vocabulary of $V = 50{,}000$ tokens. Nucleus (top-$p$) sampling with threshold $0.9$ keeps the smallest set $S$ of highest-probability tokens with $\mathbb{P}(S)\ge0.9$, then renormalizes. (a) Verify that the renormalized weights form a valid probability measure. (b) If $\mathbb{P}(S) = 0.93$ and a token $t\in S$ had $p(t) = 0.31$, what is its renormalized probability, and what measure-theoretic operation is this?

**Intuition.** Truncating and rescaling is just conditioning on the event $S$; the axioms guarantee the result is still a probability measure.

**Solution.**

*Step 1 (a).* Define $q(t) = p(t)/\mathbb{P}(S)$ for $t\in S$ and $q(t) = 0$ otherwise. Since $\mathbb{P}(S)\ge0.9 \gt 0$ and $p(t)\ge0$, Axiom 1 holds.

*Step 2 (a).* $\sum_{t\in S}q(t) = \frac{1}{\mathbb{P}(S)}\sum_{t\in S}p(t) = \frac{\mathbb{P}(S)}{\mathbb{P}(S)} = 1$, which is Axiom 2.

*Step 3 (a).* On a finite space, the probability of an event is the sum of its members' weights, so additivity (Axiom 3) is inherited from the additivity of finite sums. Hence $q$ is a probability measure on $\Omega$ supported on $S$.

*Step 4 (b).* $q(t) = 0.31/0.93 = 1/3 = 0.333333$. This is precisely conditioning: $q(\{t\}) = \mathbb{P}(\{t\}\mid S)$, the restriction of $\mathbb{P}$ to $S$ followed by renormalization.

$$
\boxed{q(t) = \frac{0.31}{0.93} = \frac13 = 0.3333}
$$

**Key takeaway.** Truncate-and-renormalize schemes (top-$k$, top-$p$) are conditional probability measures; the axioms guarantee they stay coherent.

In [15]:
# Build a Zipf-like next-token distribution, take the 0.9 nucleus, and check the axioms.
V = 50_000
w = 1.0 / (np.arange(1, V + 1) ** 1.1)
p_tok = w / w.sum()
order = np.argsort(p_tok)[::-1]
cum = np.cumsum(p_tok[order])
k = int(np.searchsorted(cum, 0.9) + 1)
S_mass = cum[k - 1]
q = p_tok[order[:k]] / S_mass
print(f"nucleus size k = {k:,} of V = {V:,}   P(S) = {S_mass:.6f}")
print(f"q >= 0 everywhere: {bool(np.all(q >= 0))}   sum q = {q.sum():.12f}")
print(f"part (b): 0.31 / 0.93 = {0.31/0.93:.6f}")
assert abs(q.sum() - 1.0) < 1e-12 and np.all(q >= 0)
assert abs(0.31 / 0.93 - 1 / 3) < 1e-12

nucleus size k = 7,293 of V = 50,000   P(S) = 0.900006
q >= 0 everywhere: True   sum q = 1.000000000000
part (b): 0.31 / 0.93 = 0.333333


### Problem L2.6 — The Born Rule as a Kolmogorov Measure (Physics)

**Statement.** A qubit in state $\lvert\psi\rangle = \alpha\lvert0\rangle + \beta\lvert1\rangle$ with $\lvert\alpha\rvert^2+\lvert\beta\rvert^2 = 1$ is measured in the computational basis. Show that the Born-rule outcome probabilities form a valid probability measure, and compute them for $\alpha = 1/\sqrt3$ and $\beta = \sqrt{2/3}\,e^{i\pi/4}$.

**Intuition.** For one fixed measurement basis the Born rule hands us squared moduli that are non-negative and sum to the squared norm of a unit vector — exactly Axioms 1 and 2.

**Solution.**

*Step 1.* Take $\Omega = \{0,1\}$, $\mathcal{F} = \{\emptyset,\{0\},\{1\},\Omega\}$, $\mathbb{P}(\{0\}) = \lvert\alpha\rvert^2$, $\mathbb{P}(\{1\}) = \lvert\beta\rvert^2$.

*Step 2.* Axiom 1: squared moduli are non-negative. Axiom 2: $\mathbb{P}(\Omega) = \lvert\alpha\rvert^2+\lvert\beta\rvert^2 = \lVert\psi\rVert^2 = 1$. Axiom 3: on a two-point space additivity over the only disjoint pair $\{0\},\{1\}$ holds by construction.

*Step 3.* For the given state, $\lvert\alpha\rvert^2 = 1/3$ and $\lvert\beta\rvert^2 = \left(\sqrt{2/3}\right)^2\lvert e^{i\pi/4}\rvert^2 = 2/3$, since the phase has unit modulus.

*Step 4.* The phase is invisible to *this* measurement, but it is not physically idle: measuring in the $\lvert\pm\rangle$ basis produces probabilities $\tfrac12 \pm \operatorname{Re}(\bar\alpha\beta)$, which do depend on it. Kolmogorov's axioms hold per observable; no single measure covers non-commuting observables jointly, which is the content of Bell's theorem.

$$
\boxed{\mathbb{P}(0) = \frac13 = 0.3333, \qquad \mathbb{P}(1) = \frac23 = 0.6667}
$$

**Key takeaway.** For each fixed measurement basis quantum mechanics hands Kolmogorov an ordinary probability measure; the strangeness lives in how measures for *different* bases interrelate.

In [16]:
alpha = 1 / np.sqrt(3)
beta = np.sqrt(2 / 3) * np.exp(1j * np.pi / 4)
P0, P1 = abs(alpha) ** 2, abs(beta) ** 2
print(f"P(0) = |alpha|^2 = {P0:.6f}   P(1) = |beta|^2 = {P1:.6f}   sum = {P0 + P1:.12f}")
plus, minus = (alpha + beta) / np.sqrt(2), (alpha - beta) / np.sqrt(2)
print(f"in the +/- basis: P(+) = {abs(plus)**2:.6f}   P(-) = {abs(minus)**2:.6f}"
      f"   sum = {abs(plus)**2 + abs(minus)**2:.12f}   (phase now matters)")
assert abs(P0 - 1 / 3) < 1e-12 and abs(P1 - 2 / 3) < 1e-12
assert abs(P0 + P1 - 1) < 1e-12 and abs(abs(plus) ** 2 + abs(minus) ** 2 - 1) < 1e-12

P(0) = |alpha|^2 = 0.333333   P(1) = |beta|^2 = 0.666667   sum = 1.000000000000
in the +/- basis: P(+) = 0.833333   P(-) = 0.166667   sum = 1.000000000000   (phase now matters)


## L3 — Challenge Proofs

### Problem L3.1 — The Matching Problem and $1/e$

**Statement.** $n$ letters are placed uniformly at random into $n$ addressed envelopes (a uniformly random permutation). Using inclusion–exclusion, find the probability of *no* correct match and its limit as $n\to\infty$.

**Intuition.** The events "letter $i$ is correct" are strongly dependent, so only the full alternating sum of Theorem 4.3 can handle them — and it collapses to a truncation of the series for $e^{-1}$.

**Solution.**

*Step 1.* Let $A_i$ = "letter $i$ lands in envelope $i$". For any fixed set of $k$ indices, the permutations fixing all $k$ number $(n-k)!$, so $\mathbb{P}(A_{i_1}\cap\cdots\cap A_{i_k}) = (n-k)!/n!$ — the same value for every $k$-set.

*Step 2.* Theorem 4.3 sums over all $\binom{n}{k}$ index sets of size $k$:

$$
\mathbb{P}\!\left(\bigcup_{i=1}^{n}A_i\right) = \sum_{k=1}^{n}(-1)^{k+1}\binom{n}{k}\frac{(n-k)!}{n!} = \sum_{k=1}^{n}\frac{(-1)^{k+1}}{k!},
$$

using $\binom nk\frac{(n-k)!}{n!} = \frac{1}{k!}$.

*Step 3.* Complement rule (Theorem 4.2):

$$
\mathbb{P}(\text{no match}) = 1 - \sum_{k=1}^{n}\frac{(-1)^{k+1}}{k!} = \sum_{k=0}^{n}\frac{(-1)^k}{k!} \xrightarrow[n\to\infty]{} e^{-1} = 0.367879.
$$

*Step 4.* The series is alternating with decreasing terms, so the error after $n$ terms is at most $1/(n+1)!$. At $n = 6$ that is $1/5040 = 1.98\times10^{-4}$, and indeed the exact value $0.368056$ differs from $e^{-1}$ by $1.76\times10^{-4}$.

$$
\boxed{\mathbb{P}(\text{derangement of }n) = \sum_{k=0}^{n}\frac{(-1)^k}{k!} \longrightarrow e^{-1} = 0.367879}
$$

**Key takeaway.** Full inclusion–exclusion tames heavily dependent events, and here the alternating factorial series is the exponential series in disguise.

In [17]:
def p_derange(n):
    return sum((-1) ** k / math.factorial(k) for k in range(n + 1))

print(f"{'n':>3} {'series':>12} {'exact !n/n!':>14} {'|.-1/e|':>12} {'1/(n+1)!':>12}")
for n in (1, 2, 3, 4, 5, 6, 8, 12):
    subfact = round(math.factorial(n) / math.e)          # !n = nearest integer to n!/e, n >= 1
    print(f"{n:>3} {p_derange(n):>12.8f} {subfact/math.factorial(n):>14.8f}"
          f" {abs(p_derange(n)-1/math.e):>12.3e} {1/math.factorial(n+1):>12.3e}")

trials, n_letters = 200_000, 6
perms = np.argsort(rng.random((trials, n_letters)), axis=1)
no_match = np.mean(~np.any(perms == np.arange(n_letters), axis=1))
print(f"\nMonte Carlo n=6: {no_match:.6f}   exact: {p_derange(6):.6f}   1/e: {1/math.e:.6f}")
assert all(abs(p_derange(n) - 1 / math.e) <= 1 / math.factorial(n + 1) for n in range(1, 13))
assert abs(no_match - p_derange(6)) < 0.01

  n       series    exact !n/n!      |.-1/e|     1/(n+1)!
  1   0.00000000     0.00000000    3.679e-01    5.000e-01
  2   0.50000000     0.50000000    1.321e-01    1.667e-01
  3   0.33333333     0.33333333    3.455e-02    4.167e-02
  4   0.37500000     0.37500000    7.121e-03    8.333e-03
  5   0.36666667     0.36666667    1.213e-03    1.389e-03
  6   0.36805556     0.36805556    1.761e-04    1.984e-04
  8   0.36788194     0.36788194    2.503e-06    2.756e-06
 12   0.36787944     0.36787944    1.498e-10    1.606e-10

Monte Carlo n=6: 0.368845   exact: 0.368056   1/e: 0.367879


### Problem L3.2 — The First Borel–Cantelli Lemma

**Statement.** Prove that if $\sum_{n=1}^{\infty}\mathbb{P}(A_n) \lt \infty$ then $\mathbb{P}(A_n \text{ infinitely often}) = 0$, where $\{A_n \text{ i.o.}\} = \bigcap_{n=1}^{\infty}\bigcup_{k=n}^{\infty}A_k$.

**Intuition.** "Infinitely often" means "in every tail"; the union bound makes each tail as small as the tail of a convergent series, and continuity lets us pass to the limit.

**Solution.**

*Step 1.* Put $B_n = \bigcup_{k\ge n}A_k$. Each $B_n\in\mathcal{F}$ by Definition 3.3, and $B_1\supseteq B_2\supseteq\cdots$ since the union runs over fewer sets as $n$ grows. By definition $\{A_n\text{ i.o.}\} = \bigcap_n B_n$.

*Step 2.* Continuity from above (Theorem 4.5) gives $\mathbb{P}(A_n\text{ i.o.}) = \lim_{n\to\infty}\mathbb{P}(B_n)$; the hypothesis $\mathbb{P}(B_1)\le1\lt\infty$ needed there is automatic for a probability measure.

*Step 3.* Theorem 4.4 applied to the countable union gives $\mathbb{P}(B_n)\le\sum_{k\ge n}\mathbb{P}(A_k)$.

*Step 4.* The right side is the tail of a convergent series, hence tends to $0$. Squeezing with Axiom 1, $\mathbb{P}(B_n)\to0$, so $\mathbb{P}(A_n\text{ i.o.}) = 0$. $\blacksquare$ Every tool used — union bound and continuity — was itself derived from the three axioms.

$$
\boxed{\sum_n \mathbb{P}(A_n) \lt \infty \implies \mathbb{P}(A_n \text{ i.o.}) = 0}
$$

**Key takeaway.** Summable failure probabilities guarantee that, almost surely, only finitely many failures ever occur — the step behind the strong law of large numbers.

In [18]:
# Summable case P(A_n) = 1/n^2 vs non-summable P(A_n) = 1/n, independent events.
N, paths = 4000, 300
n_idx = np.arange(1, N + 1)
for label, probs in (("1/n^2 (summable)", 1 / n_idx ** 2), ("1/n (not summable)", 1 / n_idx)):
    hits = rng.random((paths, N)) < probs
    total = hits.sum(axis=1)
    tail = hits[:, N // 2:].sum(axis=1)          # occurrences after index N/2
    print(f"{label:>20}: sum P(A_n) = {probs.sum():8.3f} | mean total hits = {total.mean():8.2f}"
          f" | mean hits after n={N//2} = {tail.mean():7.3f}")
assert (1 / n_idx ** 2).sum() < 2.0 and (1 / n_idx).sum() > 8.0

    1/n^2 (summable): sum P(A_n) =    1.645 | mean total hits =     1.67 | mean hits after n=2000 =   0.000
  1/n (not summable): sum P(A_n) =    8.871 | mean total hits =     8.99 | mean hits after n=2000 =   0.663


### Problem L3.3 — Finite Additivity Plus Continuity at $\emptyset$ Equals Countable Additivity

**Statement.** Let $\mathbb{P}:\mathcal{F}\to[0,1]$ be finitely additive with $\mathbb{P}(\Omega) = 1$. Prove that $\mathbb{P}$ is countably additive **if and only if** $\mathbb{P}$ is continuous at $\emptyset$, i.e. $B_n\downarrow\emptyset$ implies $\mathbb{P}(B_n)\to0$.

**Intuition.** Countable additivity says a countable sum has no leftover; continuity at $\emptyset$ says the leftover after $n$ terms vanishes. They are the same statement read from two ends.

**Solution.**

*Step 1 ($\Longleftarrow$).* Assume continuity at $\emptyset$. Let $A_1,A_2,\ldots$ be pairwise disjoint with union $A$, and define the tail remainder $B_n = A\setminus\bigcup_{k=1}^{n}A_k \in \mathcal{F}$.

*Step 2.* The $B_n$ decrease, and $\bigcap_n B_n = \emptyset$: every $\omega\in A$ lies in exactly one $A_k$, hence leaves $B_n$ as soon as $n\ge k$.

*Step 3.* Finite additivity on the disjoint decomposition $A = A_1\cup\cdots\cup A_n\cup B_n$ gives $\mathbb{P}(A) = \sum_{k=1}^{n}\mathbb{P}(A_k) + \mathbb{P}(B_n)$. Letting $n\to\infty$ and using $\mathbb{P}(B_n)\to0$,

$$
\mathbb{P}(A) = \lim_{n\to\infty}\sum_{k=1}^{n}\mathbb{P}(A_k) = \sum_{k=1}^{\infty}\mathbb{P}(A_k),
$$

which is countable additivity.

*Step 4 ($\Longrightarrow$).* Assume $\mathbb{P}$ is countably additive and let $B_n\downarrow\emptyset$. Then $B_1\setminus B_n \uparrow B_1\setminus\bigcap_k B_k = B_1$, so continuity from below (Proof 5.5, which uses only countable additivity) gives $\mathbb{P}(B_1\setminus B_n)\to \mathbb{P}(B_1)$. Finite additivity on $B_1 = (B_1\setminus B_n)\cup B_n$ gives $\mathbb{P}(B_n) = \mathbb{P}(B_1) - \mathbb{P}(B_1\setminus B_n)\to 0$. Both directions hold, so the equivalence is proved. $\blacksquare$

$$
\boxed{\text{finite additivity} + \text{continuity at } \emptyset \iff \text{countable additivity}}
$$

**Key takeaway.** Kolmogorov's third axiom is exactly a limit-compatibility requirement, which is why analysis and probability mesh at all.

The cell below exhibits both directions numerically on $\Omega = \{1,2,3,\ldots\}$ with $\mathbb{P}(\{k\}) = 2^{-k}$: the tail remainders $\mathbb{P}(B_n)$ of Step 1 vanish, and the partial sums converge to $\mathbb{P}(A)$.

In [19]:
K = 60
weights = 0.5 ** np.arange(1, K + 1)                 # P({k}) = 2^-k, truncated at K
A_mass = weights.sum()
partial = np.cumsum(weights)
remainder = A_mass - partial                          # P(B_n) of Step 3
print(f"P(Omega) (truncated at k={K}) = {A_mass:.15f}")
for n in (1, 2, 5, 10, 20, 40):
    print(f"  n={n:>2}  sum_{{k<=n}} P(A_k) = {partial[n-1]:.12f}   P(B_n) = {remainder[n-1]:.3e}")
print(f"max |P(A) - (partial + remainder)| = {np.max(np.abs(A_mass - (partial + remainder))):.3e}")
assert remainder[-1] < 1e-15
assert np.allclose(partial + remainder, A_mass, atol=1e-15)

P(Omega) (truncated at k=60) = 1.000000000000000
  n= 1  sum_{k<=n} P(A_k) = 0.500000000000   P(B_n) = 5.000e-01
  n= 2  sum_{k<=n} P(A_k) = 0.750000000000   P(B_n) = 2.500e-01
  n= 5  sum_{k<=n} P(A_k) = 0.968750000000   P(B_n) = 3.125e-02
  n=10  sum_{k<=n} P(A_k) = 0.999023437500   P(B_n) = 9.766e-04
  n=20  sum_{k<=n} P(A_k) = 0.999999046326   P(B_n) = 9.537e-07
  n=40  sum_{k<=n} P(A_k) = 0.999999999999   P(B_n) = 9.095e-13
max |P(A) - (partial + remainder)| = 0.000e+00


### Problem L3.4 — Bonferroni Inequalities (Truncated Inclusion–Exclusion)

**Statement.** Prove the first two Bonferroni inequalities for events $A_1,\ldots,A_n$: truncating inclusion–exclusion after the first-order terms overestimates, and after the second-order terms underestimates,

$$
S_1 - S_2 \le \mathbb{P}\!\left(\bigcup_{i=1}^{n}A_i\right) \le S_1, \qquad S_1 = \sum_i \mathbb{P}(A_i), \quad S_2 = \sum_{i \lt j}\mathbb{P}(A_i\cap A_j).
$$

**Intuition.** Fix an outcome lying in exactly $m$ of the events; the whole claim reduces to two elementary inequalities about the integer $m$.

**Solution.**

*Step 1.* Work pointwise. Fix $\omega$ and let $m = m(\omega)$ be the number of events containing it. Then $\mathbf{1}_{\cup A_i}(\omega) = \mathbf{1}\{m\ge1\}$, while $\sum_i\mathbf{1}_{A_i}(\omega) = m$ and $\sum_{i \lt j}\mathbf{1}_{A_i\cap A_j}(\omega) = \binom{m}{2}$.

*Step 2 (upper bound).* $\mathbf{1}\{m\ge1\}\le m$ for every integer $m\ge0$, so $\mathbf{1}_{\cup A_i}\le\sum_i\mathbf{1}_{A_i}$ pointwise.

*Step 3 (lower bound).* Claim $m - \binom m2 \le \mathbf{1}\{m\ge1\}$ for every integer $m\ge0$: at $m=0$ both sides are $0$; at $m=1$, $1-0=1$; at $m=2$, $2-1=1$; for $m\ge3$, $\binom m2 = \frac{m(m-1)}{2}\ge m$, so the left side is $\le0 \lt 1$. Hence $\sum_i\mathbf{1}_{A_i} - \sum_{i \lt j}\mathbf{1}_{A_i\cap A_j} \le \mathbf{1}_{\cup A_i}$ pointwise.

*Step 4.* Integrate both pointwise inequalities against $\mathbb{P}$. On a countable space this is a sum of the inequalities weighted by the non-negative point masses $\mathbb{P}(\{\omega\})$, legitimate by Axiom 3; in general it is monotonicity of the integral, which rests on Axiom 1. Since $\int\mathbf{1}_E\,dP = \mathbb{P}(E)$ for every $E\in\mathcal{F}$, the two displays become $\mathbb{P}(\bigcup_iA_i)\le S_1$ and $\mathbb{P}(\bigcup_iA_i)\ge S_1 - S_2$. $\blacksquare$

$$
\boxed{S_1 - S_2 \le \mathbb{P}\!\left(\bigcup_{i}A_i\right) \le S_1}
$$

**Key takeaway.** Truncations of inclusion–exclusion alternate between over- and under-estimates; the indicator method converts a set inequality into an inequality about a single integer $m$.

In [20]:
# Random finite space, random events: check S1 - S2 <= P(union) <= S1 on many instances.
worst_low = worst_high = 0.0
for _ in range(2000):
    size, n_ev = 12, 4
    w = rng.random(size); w /= w.sum()
    members = [np.flatnonzero(rng.random(size) < 0.4) for _ in range(n_ev)]
    Pm = lambda idx: w[idx].sum()
    S1 = sum(Pm(A) for A in members)
    S2 = sum(Pm(np.intersect1d(A, B)) for A, B in combinations(members, 2))
    union = Pm(np.unique(np.concatenate(members)) if any(len(A) for A in members) else np.array([], int))
    worst_low = max(worst_low, (S1 - S2) - union)
    worst_high = max(worst_high, union - S1)
print(f"worst violation of  S1 - S2 <= P(union) : {worst_low:.3e}")
print(f"worst violation of  P(union) <= S1      : {worst_high:.3e}")
assert worst_low <= 1e-12 and worst_high <= 1e-12

worst violation of  S1 - S2 <= P(union) : 3.331e-16
worst violation of  P(union) <= S1      : 0.000e+00


### Problem L3.5 — Non-Measurable Sets: Why $\mathbb{P}$ Cannot Live on Every Subset

**Statement.** Why do the axioms restrict $\mathbb{P}$ to a $\sigma$-algebra $\mathcal{F}$ instead of defining $\mathbb{P}(A)$ for *every* subset $A\subseteq\Omega$? Give the Vitali construction in full, naming the translations used and the choice principle invoked.

**Intuition.** If a probability were defined on every subset of $[0,1)$ and were translation invariant, one could cut $[0,1)$ into countably many congruent pieces — and a countable sum of one repeated constant can be neither $0$ nor $1$.

**Solution.**

*Step 1.* On a countable $\Omega$ there is no obstacle: take $\mathcal{F} = 2^{\Omega}$ and put any non-negative weights summing to 1.

*Step 2 (the construction).* On $[0,1)$ define $x\sim y \iff x-y\in\mathbb{Q}$. This is an equivalence relation, and its classes partition $[0,1)$. By the **axiom of choice**, pick exactly one representative from each class to form a set $V\subseteq[0,1)$.

*Step 3 (the translations).* For $q\in\mathbb{Q}\cap[0,1)$ let $V\oplus q = \{v+q \bmod 1 : v\in V\}$ — translation in the compact group $[0,1)$ with addition modulo 1, which is exactly the group under which the uniform measure is invariant. The family $\{V\oplus q : q\in\mathbb{Q}\cap[0,1)\}$ is countable, pairwise disjoint, and its union is $[0,1)$: given $x\in[0,1)$, let $v$ be the representative of its class; then $q = (x - v)\bmod 1$ is rational and $x\in V\oplus q$, and if $x$ lay in two such sets their representatives would differ by a rational, forcing them to be the same representative.

*Step 4 (the contradiction).* Suppose $\mathbb{P}$ were defined on all subsets, countably additive, and invariant under $\oplus q$. Then every $V\oplus q$ has the same value $c = \mathbb{P}(V)$, and countable additivity forces $1 = \mathbb{P}([0,1)) = \sum_{q}c$. If $c = 0$ the sum is $0$; if $c \gt 0$ the sum diverges. Both contradict $1$. Hence no such $\mathbb{P}$ exists on $2^{[0,1)}$, and $\mathcal{F}$ must be a strictly smaller $\sigma$-algebra — in practice the Borel sets, everything reachable by countable operations on intervals.

$$
\boxed{\text{No translation-invariant countably additive } \mathbb{P} \text{ exists on } 2^{[0,1)};\ \mathcal{F} \text{ must be restricted}}
$$

**Key takeaway.** The $\sigma$-algebra is a consistency requirement forced by countable additivity plus translation invariance — invisible in applied work, indispensable for the theory to be well-posed.

In [21]:
# The arithmetic of Step 4: a countable sum of a single constant c can never equal 1.
copies = [10 ** e for e in (1, 4, 8, 12, 16)]
print(f"{'c':>10} " + " ".join(f"{'sum over 1e' + str(int(math.log10(m))):>14}" for m in copies))
for c in (0.0, 1e-9, 1e-3, 0.5):
    print(f"{c:>10.0e} " + " ".join(f"{c * m:>14.4e}" for m in copies))
print("c = 0 gives 0 forever; every c > 0 diverges. No c makes the sum equal 1 -- the contradiction.")
assert 0.0 * 10 ** 16 == 0.0 and 1e-9 * 10 ** 16 > 1.0

         c   sum over 1e1   sum over 1e4   sum over 1e8  sum over 1e12  sum over 1e16
     0e+00     0.0000e+00     0.0000e+00     0.0000e+00     0.0000e+00     0.0000e+00
     1e-09     1.0000e-08     1.0000e-05     1.0000e-01     1.0000e+03     1.0000e+07
     1e-03     1.0000e-02     1.0000e+01     1.0000e+05     1.0000e+09     1.0000e+13
     5e-01     5.0000e+00     5.0000e+03     5.0000e+07     5.0000e+11     5.0000e+15
c = 0 gives 0 forever; every c > 0 diverges. No c makes the sum equal 1 -- the contradiction.
